# 🛒 CirklaGen Phase 2: Cirkla Tray LoRA Training Pipeline

> **Specification & Planning Notice**: This Colab notebook is an executable skeleton and configuration scaffold for training a custom packaging LoRA in Phase 2. **No models have been trained yet.** Follow the instructions below to train against your curated Cirkla tray dataset once gathered.

--- 
### Step 1: Install Dependencies & Setup Toolkit

We use [ai-toolkit by ostris](https://github.com/ostris/ai-toolkit), the industry-standard framework for training high-fidelity SDXL and FLUX LoRAs with low VRAM footprints.

> **[ACTION REQUIRED: Verify current package versions and CUDA compatibility before running]**

In [ ]:
# [ACTION REQUIRED: Verify current repository URL, commit hash, and torch/cuda versions]
!nvidia-smi

!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
%cd /content/ai-toolkit

# Install core training requirements
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install bitsandbytes torchvision accelerate diffusers huggingface_hub peft

--- 
### Step 2: Mount Google Drive

Mount your Google Drive to load the curated dataset (20–40 images with `.txt` captions) and persist your trained LoRA checkpoints.

> **[ACTION REQUIRED: Verify your Google Drive path matches your dataset directory]**

In [ ]:
# [ACTION REQUIRED: Confirm Google account authentication and Drive mount location]
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# [ACTION REQUIRED: Verify this path points to your Cirkla dataset folder]
DATASET_DIR = Path("/content/drive/MyDrive/cirklagen/dataset")
OUTPUT_DIR = Path("/content/drive/MyDrive/cirklagen/output_lora")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset Directory: {DATASET_DIR}")
print(f"Output Directory:  {OUTPUT_DIR}")

--- 
### Step 3: Dataset Verification

Verify that training pairs (`.png`/`.jpg` + `.txt`) exist and check for presence of the trigger word `cirkla_tray5`.

> **[ACTION REQUIRED: Verify every training image has a matching .txt caption with the trigger word]**

In [ ]:
# [ACTION REQUIRED: Ensure at least 20-40 image-caption pairs are present]
image_files = sorted(list(DATASET_DIR.glob("*.png")) + list(DATASET_DIR.glob("*.jpg")) + list(DATASET_DIR.glob("*.jpeg")))
print(f"Found {len(image_files)} training images in {DATASET_DIR}")

trigger_count = 0
for img_p in image_files[:5]:
    txt_p = img_p.with_suffix(".txt")
    if txt_p.is_file():
        content = txt_p.read_text(encoding="utf-8").strip()
        has_trigger = "cirkla_tray5" in content
        if has_trigger:
            trigger_count += 1
        print(f"- Sample: {img_p.name} | Trigger Present: {has_trigger}")
        print(f"  Caption: {content[:80]}...")
    else:
        print(f"- WARNING: Missing caption for {img_p.name}")

--- 
### Step 4: SDXL LoRA Training Configuration (Free Colab T4 GPU)

This configuration is tailored to train a high-quality SDXL LoRA within the **16 GB VRAM budget of a free Colab T4 instance**.
- Base model: `stabilityai/stable-diffusion-xl-base-1.0`
- Precision: `fp16` mixed precision with 8-bit AdamW
- Rank / Alpha: `16 / 16` (ideal for rigid packaging geometry without bloated file size)
- Batch size: `1` with `gradient_accumulation_steps: 4`

> **[ACTION REQUIRED: Verify base model checkpoint ID and learning rate settings]**

In [ ]:
# [ACTION REQUIRED: Verify base model path, resolution, and batch size for your GPU]
sdxl_config_text = """
job: extension
config:
  name: cirkla_tray5_sdxl_lora
  process:
    - type: sd_trainer
      training_folder: /content/drive/MyDrive/cirklagen/output_lora
      device: cuda:0
      network:
        type: lora
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 250
        max_step_saves_to_keep: 3
      datasets:
        - folder_path: /content/drive/MyDrive/cirklagen/dataset
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [1024, 1024]
      train:
        batch_size: 1
        gradient_accumulation_steps: 4
        steps: 1500
        lr: 0.0001
        optimizer: adamw8bit
        dtype: fp16
        gradient_checkpointing: true
        noise_scheduler: flowmatch
      model:
        name_or_path: stabilityai/stable-diffusion-xl-base-1.0
        is_xl: true
      sample:
        sampler: ddim
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "cirkla_tray5, front medium view, beige molded fiber meat package in refrigerated meat aisle"
          - "cirkla_tray5, side profile inspecting scalloped fluted ribs, cold store lighting"
"""

config_path_sdxl = Path("/content/ai-toolkit/config/cirkla_tray_sdxl.yaml")
config_path_sdxl.parent.mkdir(parents=True, exist_ok=True)
config_path_sdxl.write_text(sdxl_config_text, encoding="utf-8")
print(f"SDXL configuration written to: {config_path_sdxl}")

--- 
### Step 5: FLUX.1 LoRA Training Configuration (Large GPU / Cloud Instance)

When scaling up to state-of-the-art **FLUX.1-dev**, use this configuration on an **A100 (40/80 GB) or L4 (24 GB)** GPU.

> **[ACTION REQUIRED: Requires Hugging Face Access Token with approved FLUX.1-dev access and 24 GB+ VRAM]**

In [ ]:
# [ACTION REQUIRED: Verify HF_TOKEN with FLUX license acceptance and ensure GPU has >= 24GB VRAM]
flux_config_text = """
# ==============================================================================
# FLUX.1-dev LoRA Configuration (Requires A100 / L4 / H100 with >= 24GB VRAM)
# ==============================================================================
# job: extension
# config:
#   name: cirkla_tray5_flux_lora
#   process:
#     - type: sd_trainer
#       training_folder: /content/drive/MyDrive/cirklagen/output_lora_flux
#       device: cuda:0
#       network:
#         type: lora
#         linear: 16
#         linear_alpha: 16
#       save:
#         dtype: bfloat16
#         save_every: 250
#         max_step_saves_to_keep: 3
#       datasets:
#         - folder_path: /content/drive/MyDrive/cirklagen/dataset
#           caption_ext: "txt"
#           cache_latents_to_disk: true
#           resolution: [1024, 1024]
#       train:
#         batch_size: 1
#         gradient_accumulation_steps: 4
#         steps: 2000
#         lr: 0.0001
#         optimizer: adamw8bit
#         dtype: bf16
#         gradient_checkpointing: true
#       model:
#         name_or_path: black-forest-labs/FLUX.1-dev
#         is_flux: true
#         quantize: true  # Set to true on 24GB VRAM instances
# ==============================================================================
"""

config_path_flux = Path("/content/ai-toolkit/config/cirkla_tray_flux.yaml")
config_path_flux.parent.mkdir(parents=True, exist_ok=True)
config_path_flux.write_text(flux_config_text, encoding="utf-8")
print(f"FLUX configuration template saved to: {config_path_flux}")

--- 
### Step 6: Launch LoRA Training Run

Execute the training job. On a free Colab T4 GPU, training SDXL for 1500 steps typically takes approximately **45 to 65 minutes**.

> **[ACTION REQUIRED: Verify GPU memory allocation and watch first 50 steps for NaN loss or VRAM out-of-memory]**

In [ ]:
# [ACTION REQUIRED: Verify active working directory and config file path]
%cd /content/ai-toolkit

# Execute SDXL Training
!python run.py config/cirkla_tray_sdxl.yaml

--- 
### Step 7: Inference Verification with Trained LoRA

Test the fine-tuned weights using Diffusers and verify the trigger token `cirkla_tray5`.

> **[ACTION REQUIRED: Update checkpoint file name to point to your saved .safetensors file]**

In [ ]:
# [ACTION REQUIRED: Verify the exact path to your trained .safetensors file]
import torch
from diffusers import AutoPipelineForText2Image

lora_weights_path = "/content/drive/MyDrive/cirklagen/output_lora/cirkla_tray5_sdxl_lora.safetensors"

if os.path.exists(lora_weights_path):
    pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16"
    ).to("cuda")
    
    pipe.load_lora_weights(lora_weights_path)
    
    test_prompt = (
        "cirkla_tray5, candid phone photo of shopper holding beige molded plant fiber beef tray "
        "with vertical fluted ribs in refrigerated grocery meat aisle"
    )
    
    image = pipe(test_prompt, num_inference_steps=30, guidance_scale=6.5).images[0]
    display(image)
    image.save("/content/sample_lora_test.png")
    print("Test image generated and saved to /content/sample_lora_test.png")
else:
    print(f"Checkpoint not found at: {lora_weights_path}. Run training first.")